# Nettoyage, feature engineering et préparation des datasets finaux

Ce notebook a pour objectif de construire les datasets finaux de modélisation à partir de `application_train.csv` et `application_test.csv`.

## Objectifs

Dans cette étape, on cherche à :

- charger les tables principales et secondaires
- appliquer un nettoyage minimal utile avant agrégation
- créer quelques variables dérivées sur les tables principales
- agréger les tables secondaires par `SK_ID_CURR`
- fusionner les agrégations avec `application_train` et `application_test`
- produire deux datasets finaux :
  - `train_final`
  - `test_final`

## Remarque

Dans ce projet, le choix a été fait de travailler avec **pandas** et non PySpark, car le volume de données reste gérable dans le cadre du projet et cette approche permet d'aller plus vite vers un pipeline cohérent.

## 1. Import des bibliothèques

In [1]:
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

## 2. Fonctions utilitaires

Deux fonctions sont utilisées :

- `reduce_mem_usage()` pour réduire l’empreinte mémoire des tables volumineuses
- `flatten_columns()` pour simplifier les noms de colonnes après agrégation

In [8]:
def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype

        if str(col_type)[:3] == "int":
            c_min = df[col].min()
            c_max = df[col].max()

            if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)

        elif str(col_type)[:5] == "float":
            c_min = df[col].min()
            c_max = df[col].max()

            if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                df[col] = df[col].astype(np.float16)
            elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)

    return df


def flatten_columns(df, prefix, key_cols=("SK_ID_CURR", "SK_ID_BUREAU", "SK_ID_PREV")):
    new_cols = []
    for col in df.columns:
        if col in key_cols:
            new_cols.append(col)
        elif isinstance(col, tuple):
            parts = [str(c) for c in col if c != ""]
            new_cols.append(f"{prefix}_{'_'.join(parts).upper()}")
        else:
            new_cols.append(f"{prefix}_{str(col).upper()}")
    df.columns = new_cols
    return df

## 3. Chargement des tables principales

In [9]:
application_train = pd.read_csv(DATA_RAW / "application_train.csv")
application_test = pd.read_csv(DATA_RAW / "application_test.csv")

print("application_train :", application_train.shape)
print("application_test  :", application_test.shape)

application_train : (307511, 122)
application_test  : (48744, 121)


## 4. Création de variables simples sur les tables principales

Avant d’intégrer les tables secondaires, on crée quelques variables utiles directement à partir de `application_train` et `application_test`.

Exemples :
- âge en années
- ratio crédit / revenu
- ratio annuité / revenu
- ratio emploi / âge

In [10]:
for df in [application_train, application_test]:
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

    df["AGE_YEARS"] = (-df["DAYS_BIRTH"] / 365.25).round(2)
    df["EMPLOYED_TO_AGE_RATIO"] = df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"]

    df["CREDIT_TO_INCOME_RATIO"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    df["ANNUITY_TO_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    df["CREDIT_TO_GOODS_RATIO"] = df["AMT_CREDIT"] / df["AMT_GOODS_PRICE"].replace(0, np.nan)

## 5. Initialisation des datasets finaux

In [11]:
train_final = application_train.copy()
test_final = application_test.copy()

print("train_final :", train_final.shape)
print("test_final  :", test_final.shape)

train_final : (307511, 127)
test_final  : (48744, 126)


## 6. Agrégation de `bureau.csv`

Cette table contient des informations sur les crédits externes des clients.

On agrège ici quelques variables simples :
- historique temporel du crédit
- montants de crédit
- dettes
- montants en retard

In [12]:
bureau = pd.read_csv(DATA_RAW / "bureau.csv")
bureau = reduce_mem_usage(bureau)

bureau_agg = bureau.groupby("SK_ID_CURR").agg({
    "DAYS_CREDIT": ["mean", "min", "max"],
    "CREDIT_DAY_OVERDUE": ["mean", "max"],
    "AMT_CREDIT_SUM": ["mean", "sum", "max"],
    "AMT_CREDIT_SUM_DEBT": ["mean", "sum"],
    "AMT_CREDIT_SUM_OVERDUE": ["mean", "sum"]
}).reset_index()

bureau_agg = flatten_columns(bureau_agg, "BUREAU")
bureau_agg = bureau_agg.rename(columns={"BUREAU_SK_ID_CURR": "SK_ID_CURR"})

bureau_agg.head()

,SK_ID_CURR,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_AMT_CREDIT_SUM_MEAN,BUREAU_AMT_CREDIT_SUM_SUM,BUREAU_AMT_CREDIT_SUM_MAX,BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,BUREAU_AMT_CREDIT_SUM_DEBT_SUM,BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN,BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM
0,100001,-735.0000,-1572,-49,0.0000,0,"207,623.5781","1,453,365.0000","378,000.0000","85,240.9297","596,686.5000",0.0000,0.0000
1,100002,-874.0000,-1437,-103,0.0000,0,"108,131.9453","865,055.5625","450,000.0000","49,156.1992","245,781.0000",0.0000,0.0000
2,100003,"-1,400.7500",-2586,-606,0.0000,0,"254,350.1250","1,017,400.5000","810,000.0000",0.0000,0.0000,0.0000,0.0000
3,100004,-867.0000,-1326,-408,0.0000,0,"94,518.8984","189,037.7969","94,537.7969",0.0000,0.0000,0.0000,0.0000
4,100005,-190.6667,-373,-62,0.0000,0,"219,042.0000","657,126.0000","568,800.0000","189,469.5000","568,408.5000",0.0000,0.0000


In [13]:
train_final = train_final.merge(bureau_agg, on="SK_ID_CURR", how="left")
test_final = test_final.merge(bureau_agg, on="SK_ID_CURR", how="left")

print("Après bureau :", train_final.shape, test_final.shape)

del bureau, bureau_agg
gc.collect()

Après bureau : (307511, 139) (48744, 138)


214

## 7. Agrégation de `bureau_balance.csv`

Le dataset `bureau_balance` est au niveau `SK_ID_BUREAU`.  
Il faut donc d’abord le relier à `bureau.csv`, puis agréger au niveau client (`SK_ID_CURR`).

In [14]:
bureau = pd.read_csv(DATA_RAW / "bureau.csv", usecols=["SK_ID_BUREAU", "SK_ID_CURR"])
bureau_balance = pd.read_csv(DATA_RAW / "bureau_balance.csv")

bureau_balance_merged = bureau_balance.merge(
    bureau,
    on="SK_ID_BUREAU",
    how="left"
)

bureau_balance_agg = bureau_balance_merged.groupby("SK_ID_CURR").agg({
    "MONTHS_BALANCE": ["min", "max"]
}).reset_index()

bureau_balance_agg = flatten_columns(bureau_balance_agg, "BUREAU_BAL")
bureau_balance_agg = bureau_balance_agg.rename(columns={"BUREAU_BAL_SK_ID_CURR": "SK_ID_CURR"})

bureau_balance_agg.head()

,SK_ID_CURR,BUREAU_BAL_MONTHS_BALANCE_MIN,BUREAU_BAL_MONTHS_BALANCE_MAX
0,"100,001.0000",-51,0
1,"100,002.0000",-47,0
2,"100,005.0000",-12,0
3,"100,010.0000",-90,-2
4,"100,013.0000",-68,0


In [15]:
train_final = train_final.merge(bureau_balance_agg, on="SK_ID_CURR", how="left")
test_final = test_final.merge(bureau_balance_agg, on="SK_ID_CURR", how="left")

print("Après bureau_balance :", train_final.shape, test_final.shape)

del bureau, bureau_balance, bureau_balance_merged, bureau_balance_agg
gc.collect()

Après bureau_balance : (307511, 141) (48744, 140)


0

## 8. Agrégation de `previous_application.csv`

Cette table contient l’historique des demandes précédentes.

On agrège quelques variables représentant :
- les montants demandés
- les montants accordés
- l’annuité
- le prix du bien
- l’heure de traitement

In [16]:
prev = pd.read_csv(DATA_RAW / "previous_application.csv")
prev = reduce_mem_usage(prev)

prev_agg = prev.groupby("SK_ID_CURR").agg({
    "AMT_APPLICATION": ["mean", "max"],
    "AMT_CREDIT": ["mean", "max"],
    "AMT_ANNUITY": ["mean"],
    "AMT_GOODS_PRICE": ["mean"],
    "HOUR_APPR_PROCESS_START": ["mean"]
}).reset_index()

prev_agg = flatten_columns(prev_agg, "PREV")
prev_agg = prev_agg.rename(columns={"PREV_SK_ID_CURR": "SK_ID_CURR"})

prev_agg.head()

,SK_ID_CURR,PREV_AMT_APPLICATION_MEAN,PREV_AMT_APPLICATION_MAX,PREV_AMT_CREDIT_MEAN,PREV_AMT_CREDIT_MAX,PREV_AMT_ANNUITY_MEAN,PREV_AMT_GOODS_PRICE_MEAN,PREV_HOUR_APPR_PROCESS_START_MEAN
0,100001,"24,835.5000","24,835.5000","23,787.0000","23,787.0000","3,951.0000","24,835.5000",13.0000
1,100002,"179,055.0000","179,055.0000","179,055.0000","179,055.0000","9,251.7754","179,055.0000",9.0000
2,100003,"435,436.5000","900,000.0000","484,191.0000","1,035,882.0000","56,553.9883","435,436.5000",14.6667
3,100004,"24,282.0000","24,282.0000","20,106.0000","20,106.0000","5,357.2500","24,282.0000",5.0000
4,100005,"22,308.7500","44,617.5000","20,076.7500","40,153.5000","4,813.2002","44,617.5000",10.5000


In [17]:
train_final = train_final.merge(prev_agg, on="SK_ID_CURR", how="left")
test_final = test_final.merge(prev_agg, on="SK_ID_CURR", how="left")

print("Après previous_application :", train_final.shape, test_final.shape)

del prev, prev_agg
gc.collect()

Après previous_application : (307511, 148) (48744, 147)


0

## 9. Agrégation de `installments_payments.csv`

Cette table permet de construire des variables liées aux paiements :

- montant payé
- montant attendu
- différence entre paiement et échéance
- retard de paiement
- pourcentage payé

Une précaution est prise pour éviter les divisions par zéro.

In [18]:
ins = pd.read_csv(DATA_RAW / "installments_payments.csv")
ins = reduce_mem_usage(ins)

ins["PAYMENT_PERC"] = ins["AMT_PAYMENT"] / ins["AMT_INSTALMENT"].replace(0, np.nan)
ins["PAYMENT_DIFF"] = ins["AMT_PAYMENT"] - ins["AMT_INSTALMENT"]
ins["PAYMENT_DELAY"] = ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]

ins_agg = ins.groupby("SK_ID_CURR").agg({
    "AMT_INSTALMENT": ["mean", "sum"],
    "AMT_PAYMENT": ["mean", "sum"],
    "PAYMENT_PERC": ["mean"],
    "PAYMENT_DIFF": ["mean"],
    "PAYMENT_DELAY": ["mean", "max"]
}).reset_index()

ins_agg = flatten_columns(ins_agg, "INST")
ins_agg = ins_agg.rename(columns={"INST_SK_ID_CURR": "SK_ID_CURR"})

ins_agg.head()

,SK_ID_CURR,INST_AMT_INSTALMENT_MEAN,INST_AMT_INSTALMENT_SUM,INST_AMT_PAYMENT_MEAN,INST_AMT_PAYMENT_SUM,INST_PAYMENT_PERC_MEAN,INST_PAYMENT_DIFF_MEAN,INST_PAYMENT_DELAY_MEAN,INST_PAYMENT_DELAY_MAX
0,100001,"5,885.1323","41,195.9258","5,885.1323","41,195.9258",1.0000,0.0000,-7.4286,10.0000
1,100002,"11,559.2471","219,625.7031","11,559.2471","219,625.7031",1.0000,0.0000,-20.4211,-12.0000
2,100003,"64,754.5859","1,618,864.6250","64,754.5859","1,618,864.6250",1.0000,0.0000,-7.2000,-2.0000
3,100004,"7,096.1548","21,288.4648","7,096.1548","21,288.4648",1.0000,0.0000,-7.6667,-3.0000
4,100005,"6,240.2051","56,161.8438","6,240.2051","56,161.8438",1.0000,0.0000,-23.5556,1.0000


In [19]:
train_final = train_final.merge(ins_agg, on="SK_ID_CURR", how="left")
test_final = test_final.merge(ins_agg, on="SK_ID_CURR", how="left")

print("Après installments_payments :", train_final.shape, test_final.shape)

del ins, ins_agg
gc.collect()

Après installments_payments : (307511, 156) (48744, 155)


0

## 10. Agrégation de `credit_card_balance.csv`

Cette table donne des informations sur l’historique des cartes de crédit :

- solde
- limite de crédit
- montants tirés
- paiements
- retards

In [20]:
ccb = pd.read_csv(DATA_RAW / "credit_card_balance.csv")
ccb = reduce_mem_usage(ccb)

ccb_agg = ccb.groupby("SK_ID_CURR").agg({
    "AMT_BALANCE": ["mean", "max"],
    "AMT_CREDIT_LIMIT_ACTUAL": ["mean", "max"],
    "AMT_DRAWINGS_CURRENT": ["mean", "sum"],
    "AMT_PAYMENT_TOTAL_CURRENT": ["mean", "sum"],
    "SK_DPD": ["mean", "max"],
    "SK_DPD_DEF": ["mean", "max"]
}).reset_index()

ccb_agg = flatten_columns(ccb_agg, "CC")
ccb_agg = ccb_agg.rename(columns={"CC_SK_ID_CURR": "SK_ID_CURR"})

ccb_agg.head()

,SK_ID_CURR,CC_AMT_BALANCE_MEAN,CC_AMT_BALANCE_MAX,CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN,CC_AMT_CREDIT_LIMIT_ACTUAL_MAX,CC_AMT_DRAWINGS_CURRENT_MEAN,CC_AMT_DRAWINGS_CURRENT_SUM,CC_AMT_PAYMENT_TOTAL_CURRENT_MEAN,CC_AMT_PAYMENT_TOTAL_CURRENT_SUM,CC_SK_DPD_MEAN,CC_SK_DPD_MAX,CC_SK_DPD_DEF_MEAN,CC_SK_DPD_DEF_MAX
0,100006,0.0000,0.0000,"270,000.0000",270000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0.0000,0
1,100011,"54,482.1133","189,000.0000","164,189.1892",180000,"2,432.4324","180,000.0000","4,520.0674","334,485.0000",0.0000,0,0.0000,0
2,100013,"18,159.9199","161,420.2188","131,718.7500",157500,"5,953.1250","571,500.0000","6,817.1724","654,448.5625",0.0104,1,0.0104,1
3,100021,0.0000,0.0000,"675,000.0000",675000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0.0000,0
4,100023,0.0000,0.0000,"135,000.0000",225000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0.0000,0


In [21]:
train_final = train_final.merge(ccb_agg, on="SK_ID_CURR", how="left")
test_final = test_final.merge(ccb_agg, on="SK_ID_CURR", how="left")

print("Après credit_card_balance :", train_final.shape, test_final.shape)

del ccb, ccb_agg
gc.collect()

Après credit_card_balance : (307511, 168) (48744, 167)


0

## 11. Agrégation de `POS_CASH_balance.csv`

Cette table contient des informations sur les échéances restantes et les retards pour des crédits de type POS / cash.

On agrège ici :
- l’historique mensuel
- le nombre d’échéances
- les retards

In [22]:
pos = pd.read_csv(DATA_RAW / "POS_CASH_balance.csv")
pos = reduce_mem_usage(pos)

pos_agg = pos.groupby("SK_ID_CURR").agg({
    "MONTHS_BALANCE": ["min", "max"],
    "CNT_INSTALMENT": ["mean", "max"],
    "CNT_INSTALMENT_FUTURE": ["mean", "max"],
    "SK_DPD": ["mean", "max"],
    "SK_DPD_DEF": ["mean", "max"]
}).reset_index()

pos_agg = flatten_columns(pos_agg, "POS")
pos_agg = pos_agg.rename(columns={"POS_SK_ID_CURR": "SK_ID_CURR"})

pos_agg.head()

,SK_ID_CURR,POS_MONTHS_BALANCE_MIN,POS_MONTHS_BALANCE_MAX,POS_CNT_INSTALMENT_MEAN,POS_CNT_INSTALMENT_MAX,POS_CNT_INSTALMENT_FUTURE_MEAN,POS_CNT_INSTALMENT_FUTURE_MAX,POS_SK_DPD_MEAN,POS_SK_DPD_MAX,POS_SK_DPD_DEF_MEAN,POS_SK_DPD_DEF_MAX
0,100001,-96,-53,4.0000,4.0000,1.4444,4.0000,0.7778,7,0.7778,7
1,100002,-19,-1,24.0000,24.0000,15.0000,24.0000,0.0000,0,0.0000,0
2,100003,-77,-18,10.1071,12.0000,5.7857,12.0000,0.0000,0,0.0000,0
3,100004,-27,-24,3.7500,4.0000,2.2500,4.0000,0.0000,0,0.0000,0
4,100005,-25,-15,11.7000,12.0000,7.2000,12.0000,0.0000,0,0.0000,0


In [23]:
train_final = train_final.merge(pos_agg, on="SK_ID_CURR", how="left")
test_final = test_final.merge(pos_agg, on="SK_ID_CURR", how="left")

print("Après POS_CASH_balance :", train_final.shape, test_final.shape)

del pos, pos_agg
gc.collect()

Après POS_CASH_balance : (307511, 178) (48744, 177)


0

## 12. Gestion des valeurs manquantes après les merges

Après la fusion des tables agrégées, certaines colonnes contiennent des valeurs manquantes.

Cela est normal : certains clients n’ont pas d’historique dans certaines tables secondaires.

Dans cette première version, une stratégie simple est utilisée :
- les colonnes numériques sont imputées à `0`
- les colonnes catégorielles restent inchangées à ce stade

In [24]:
numeric_cols_train = train_final.select_dtypes(include=[np.number]).columns
numeric_cols_test = test_final.select_dtypes(include=[np.number]).columns

train_final[numeric_cols_train] = train_final[numeric_cols_train].fillna(0)
test_final[numeric_cols_test] = test_final[numeric_cols_test].fillna(0)

print("Valeurs manquantes restantes dans train_final :", train_final.isnull().sum().sum())
print("Valeurs manquantes restantes dans test_final  :", test_final.isnull().sum().sum())

Valeurs manquantes restantes dans train_final : 764371
Valeurs manquantes restantes dans test_final  : 119034


## 13. Vérification finale des datasets

In [25]:
summary_final = pd.DataFrame({
    "dataset": ["train_final", "test_final"],
    "n_lignes": [train_final.shape[0], test_final.shape[0]],
    "n_colonnes": [train_final.shape[1], test_final.shape[1]],
    "nb_valeurs_manquantes": [
        int(train_final.isnull().sum().sum()),
        int(test_final.isnull().sum().sum())
    ]
})

summary_final

,dataset,n_lignes,n_colonnes,nb_valeurs_manquantes
0,train_final,307511,178,764371
1,test_final,48744,177,119034


In [26]:
print("Aperçu train_final")
display(train_final.head())

print("Aperçu test_final")
display(test_final.head())

Aperçu train_final


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,AGE_YEARS,EMPLOYED_TO_AGE_RATIO,CREDIT_TO_INCOME_RATIO,ANNUITY_TO_INCOME_RATIO,CREDIT_TO_GOODS_RATIO,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_AMT_CREDIT_SUM_MEAN,BUREAU_AMT_CREDIT_SUM_SUM,BUREAU_AMT_CREDIT_SUM_MAX,BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,BUREAU_AMT_CREDIT_SUM_DEBT_SUM,BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN,BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM,BUREAU_BAL_MONTHS_BALANCE_MIN,BUREAU_BAL_MONTHS_BALANCE_MAX,PREV_AMT_APPLICATION_MEAN,PREV_AMT_APPLICATION_MAX,PREV_AMT_CREDIT_MEAN,PREV_AMT_CREDIT_MAX,PREV_AMT_ANNUITY_MEAN,PREV_AMT_GOODS_PRICE_MEAN,PREV_HOUR_APPR_PROCESS_START_MEAN,INST_AMT_INSTALMENT_MEAN,INST_AMT_INSTALMENT_SUM,INST_AMT_PAYMENT_MEAN,INST_AMT_PAYMENT_SUM,INST_PAYMENT_PERC_MEAN,INST_PAYMENT_DIFF_MEAN,INST_PAYMENT_DELAY_MEAN,INST_PAYMENT_DELAY_MAX,CC_AMT_BALANCE_MEAN,CC_AMT_BALANCE_MAX,CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN,CC_AMT_CREDIT_LIMIT_ACTUAL_MAX,CC_AMT_DRAWINGS_CURRENT_MEAN,CC_AMT_DRAWINGS_CURRENT_SUM,CC_AMT_PAYMENT_TOTAL_CURRENT_MEAN,CC_AMT_PAYMENT_TOTAL_CURRENT_SUM,CC_SK_DPD_MEAN,CC_SK_DPD_MAX,CC_SK_DPD_DEF_MEAN,CC_SK_DPD_DEF_MAX,POS_MONTHS_BALANCE_MIN,POS_MONTHS_BALANCE_MAX,POS_CNT_INSTALMENT_MEAN,POS_CNT_INSTALMENT_MAX,POS_CNT_INSTALMENT_FUTURE_MEAN,POS_CNT_INSTALMENT_FUTURE_MAX,POS_SK_DPD_MEAN,POS_SK_DPD_MAX,POS_SK_DPD_DEF_MEAN,POS_SK_DPD_DEF_MAX
0,100002,1,Cash loans,M,N,Y,0,"202,500.0000","406,597.5000","24,700.5000","351,000.0000",Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.0188,-9461,-637.0000,"-3,648.0000",-2120,0.0000,1,1,0,1,1,0,Laborers,1.0000,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.0830,0.2629,0.1394,0.0247,0.0369,0.9722,0.6192,0.0143,0.0000,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.000

Aperçu test_final


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,AGE_YEARS,EMPLOYED_TO_AGE_RATIO,CREDIT_TO_INCOME_RATIO,ANNUITY_TO_INCOME_RATIO,CREDIT_TO_GOODS_RATIO,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_AMT_CREDIT_SUM_MEAN,BUREAU_AMT_CREDIT_SUM_SUM,BUREAU_AMT_CREDIT_SUM_MAX,BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,BUREAU_AMT_CREDIT_SUM_DEBT_SUM,BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN,BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM,BUREAU_BAL_MONTHS_BALANCE_MIN,BUREAU_BAL_MONTHS_BALANCE_MAX,PREV_AMT_APPLICATION_MEAN,PREV_AMT_APPLICATION_MAX,PREV_AMT_CREDIT_MEAN,PREV_AMT_CREDIT_MAX,PREV_AMT_ANNUITY_MEAN,PREV_AMT_GOODS_PRICE_MEAN,PREV_HOUR_APPR_PROCESS_START_MEAN,INST_AMT_INSTALMENT_MEAN,INST_AMT_INSTALMENT_SUM,INST_AMT_PAYMENT_MEAN,INST_AMT_PAYMENT_SUM,INST_PAYMENT_PERC_MEAN,INST_PAYMENT_DIFF_MEAN,INST_PAYMENT_DELAY_MEAN,INST_PAYMENT_DELAY_MAX,CC_AMT_BALANCE_MEAN,CC_AMT_BALANCE_MAX,CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN,CC_AMT_CREDIT_LIMIT_ACTUAL_MAX,CC_AMT_DRAWINGS_CURRENT_MEAN,CC_AMT_DRAWINGS_CURRENT_SUM,CC_AMT_PAYMENT_TOTAL_CURRENT_MEAN,CC_AMT_PAYMENT_TOTAL_CURRENT_SUM,CC_SK_DPD_MEAN,CC_SK_DPD_MAX,CC_SK_DPD_DEF_MEAN,CC_SK_DPD_DEF_MAX,POS_MONTHS_BALANCE_MIN,POS_MONTHS_BALANCE_MAX,POS_CNT_INSTALMENT_MEAN,POS_CNT_INSTALMENT_MAX,POS_CNT_INSTALMENT_FUTURE_MEAN,POS_CNT_INSTALMENT_FUTURE_MAX,POS_SK_DPD_MEAN,POS_SK_DPD_MAX,POS_SK_DPD_DEF_MEAN,POS_SK_DPD_DEF_MAX
0,100001,Cash loans,F,N,Y,0,"135,000.0000","568,800.0000","20,560.5000","450,000.0000",Unaccompanied,Working,Higher education,Married,House / apartment,0.0188,-19241,"-2,329.0000","-5,170.0000",-812,0.0000,1,1,0,1,0,1,NaN,2.0000,2,2,TUESDAY,18,0,0,0,0,0,0,Kindergarten,0.7526,0.7897,0.1595,0.0660,0.0590,0.9732,0.0000,0.0000,0.0000,0.1379,0.1250,0.0000,0.0000,0.0000,0.0505,0.0000,0.0000,0.0672,0.0612,0.9732,0.0000,0.0000,0.00

## 14. Sauvegarde des datasets finaux

Cette étape permet d’enregistrer les datasets préparés afin de les réutiliser directement dans la phase de modélisation.

In [27]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

train_final.to_csv(DATA_PROCESSED / "train_final.csv", index=False)
test_final.to_csv(DATA_PROCESSED / "test_final.csv", index=False)

print("Fichiers enregistrés dans data/processed/")

Fichiers enregistrés dans data/processed/


## 15. Conclusion

Cette étape a permis de :

- préparer les tables principales
- créer quelques variables dérivées utiles
- agréger les informations issues des datasets secondaires
- fusionner ces informations avec `application_train` et `application_test`
- construire deux datasets finaux prêts pour la modélisation

Les prochaines étapes seront :

1. séparer les variables explicatives et la cible
2. encoder les variables catégorielles
3. entraîner un modèle baseline
4. produire un tableau de bord des métriques sur train, validation et test